# SentinelIQ — 03 Autoencoder
Reconstruction-based anomaly detection on system metrics using PyTorch.

In [ ]:
!git clone https://github.com/hasan-rajab/SentinelIQ.git 2>/dev/null || echo "Already cloned"
%cd /kaggle/working/SentinelIQ
import sys
sys.path.insert(0, '/kaggle/working/SentinelIQ')
!pip install pyyaml torch scikit-learn -q


In [ ]:
!python data/simulated/pipeline.py --duration 120 --anomaly-rate 0.08


In [ ]:
import json
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay
import torch

from ml.models.autoencoder import SentinelAutoencoder

sns.set_theme(style="darkgrid")
print(f"GPU available: {torch.cuda.is_available()}")

with open('configs/model_config.yaml') as f:
    cfg = yaml.safe_load(f)

def load_jsonl(path):
    return pd.DataFrame([json.loads(l) for l in open(path)])


## Load & Split Data

In [ ]:
metrics_df = load_jsonl('data/simulated/metrics.jsonl')
feature_cols = cfg['isolation_forest']['features']
ae_cfg = {**cfg['autoencoder'], 'input_dim': len(feature_cols)}

y = metrics_df['is_anomaly'].astype(int).values

X_train, X_test, y_train, y_test = train_test_split(
    metrics_df, y, test_size=0.2, random_state=42, stratify=y)

X_val, X_test_final, y_val, y_test_final = train_test_split(
    X_test, y_test, test_size=0.5, random_state=42, stratify=y_test)

X_train_normal = X_train[y_train == 0]
print(f"Train (normal only): {len(X_train_normal)}")
print(f"Val  : {len(X_val)} | Test: {len(X_test_final)}")


## Train Autoencoder

In [ ]:
ae = SentinelAutoencoder(config=ae_cfg, feature_cols=feature_cols)
ae.fit(X_train_normal, val_df=X_val)


## Evaluate

In [ ]:
results = ae.evaluate(X_test_final, y_test_final)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

scores = ae.score(X_test_final)
RocCurveDisplay.from_predictions(y_test_final, scores, ax=axes[0], name='Autoencoder')
axes[0].set_title(f"ROC Curve (AUC={results['roc_auc']})")

PrecisionRecallDisplay.from_predictions(y_test_final, scores, ax=axes[1], name='Autoencoder')
axes[1].set_title(f"Precision-Recall (AP={results['avg_precision']})")

axes[2].hist(scores[y_test_final==0], bins=30, alpha=0.6, color='#2ecc71', label='Normal')
axes[2].hist(scores[y_test_final==1], bins=30, alpha=0.6, color='#e74c3c', label='Anomaly')
axes[2].axvline(ae.threshold, color='navy', linestyle='--', label=f'Threshold={ae.threshold:.5f}')
axes[2].set_title('Reconstruction Error Distribution')
axes[2].legend()

plt.suptitle('Autoencoder — Metrics Anomaly Detection', fontsize=14)
plt.tight_layout()
plt.show()


## Latent Space Visualization

In [ ]:
import torch
from sklearn.decomposition import PCA

ae.net.eval()
X_scaled = ae.scaler.transform(metrics_df[feature_cols].fillna(0).values)
X_tensor = torch.FloatTensor(X_scaled).to(ae.device)

with torch.no_grad():
    latent = ae.net.encode(X_tensor).cpu().numpy()

pca = PCA(n_components=2)
latent_2d = pca.fit_transform(latent)

plt.figure(figsize=(8, 6))
colors = ['#e74c3c' if a else '#2ecc71' for a in metrics_df['is_anomaly']]
plt.scatter(latent_2d[:, 0], latent_2d[:, 1], c=colors, alpha=0.5, s=20)
plt.title('Autoencoder Latent Space (PCA 2D)')
plt.xlabel('PC1')
plt.ylabel('PC2')
from matplotlib.patches import Patch
plt.legend(handles=[Patch(color='#2ecc71', label='Normal'), Patch(color='#e74c3c', label='Anomaly')])
plt.tight_layout()
plt.show()


## Save Model

In [ ]:
import os
os.makedirs('ml/saved_models', exist_ok=True)
ae.save('ml/saved_models', name='autoencoder_metrics')
print("Autoencoder saved.")
